# 🎬 Netflix Data Science Project
## Milestone #1: Data Preparation & Cleaning

---

### 🎯 **Learning Objectives**
By the end of this project, you will be able to:
- ✅ Load and inspect real-world datasets
- ✅ Identify and handle missing values strategically
- ✅ Process multi-value fields and categorical data
- ✅ Convert data types appropriately
- ✅ Detect and remove duplicates
- ✅ Ask insightful questions about data

---

### 📺 **The Business Question**

**"What makes Netflix content successful, and what patterns exist in their catalog strategy?"**

This project uses the Netflix Movies and TV Shows dataset to explore:
- Content production trends over time
- Genre preferences and distribution
- Content origination by country
- Rating classifications and their patterns
- Data quality challenges in real-world datasets

---

### 📊 **Dataset Overview**

| Column | Description | Data Quality Challenge |
|--------|-------------|------------------------|
| **show_id** | Unique identifier | ✅ Complete |
| **type** | Movie or TV Show | ✅ Complete |
| **title** | Content title | ✅ Complete |
| **director** | Director name(s) | ⚠️ 30% missing (docs/limited series) |
| **cast** | Actor names | ⚠️ 9% missing |
| **country** | Production country | ⚠️ 9% missing (international co-productions) |
| **date_added** | When added to Netflix | ⚠️ <1% missing |
| **release_year** | Original release year | ✅ Complete |
| **rating** | Content rating (G, PG, TV-MA) | ⚠️ <1% missing |
| **duration** | Length (minutes or seasons) | ⚠️ <1% missing |
| **listed_in** | Genre categories | ✅ Complete (comma-separated) |
| **description** | Plot summary | ✅ Complete |

---

## 📥 Step 1: Load & Explore the Dataset

Let's start by loading the Netflix dataset and taking a first look at what we're working with.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

In [ ]:
# Load the Netflix dataset
url = 'https://raw.githubusercontent.com/maudem-data/DATA-101/main/datasets/netflix_titles.csv'

try:
    df = pd.read_csv(url)
    print(f"✅ Dataset loaded successfully!")
    print(f"\n📊 Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")

# Display first few rows
print("\n" + "="*100)
print("FIRST 5 ROWS")
print("="*100)
df.head()

### 🤔 **Before You Continue: Make a Prediction!**

Look at the first few rows above. Make your predictions:

1. **What do you notice about missing values?** Which columns have NaN?
2. **What's the ratio of Movies to TV Shows?** (You'll see this below)
3. **Why might the 'director' column be missing for some entries?**

Write your thoughts below before scrolling to see the answers!

---

## 🔍 Step 2: Initial Data Inspection

Now let's examine the data structure, types, and get descriptive statistics.

In [ ]:
print("\n" + "="*100)
print("DATA TYPES & NON-NULL COUNTS")
print("="*100)
df.info(verbose=True)

print("\n" + "="*100)
print("DESCRIPTIVE STATISTICS (Numerical Columns)")
print("="*100)
df.describe()

### 💡 **Key Observations from Initial Inspection:**

- **Release Year Range**: Content from 1925 to 2021 (mean: ~2014)
- **Content Type**: Mostly movies (6,131) vs TV shows (2,676)
- **Duration**: Movies average ~100 minutes; TV shows average ~1.7 seasons
- **Timeline**: Most content added between 2018-2021 (Netflix expansion period)

In [ ]:
# Let's look at categorical data too
print("\n" + "="*100)
print("CATEGORICAL DATA SUMMARY")
print("="*100)

print("\n📺 CONTENT TYPE DISTRIBUTION:")
print(df['type'].value_counts())
print(f"Ratio: {df[df['type']=='Movie'].shape[0]}/{df[df['type']=='TV Show'].shape[0]} "
      f"= {df[df['type']=='Movie'].shape[0]/df[df['type']=='TV Show'].shape[0]:.2f}x more movies")

print("\n⭐ TOP 10 CONTENT RATINGS:")
print(df['rating'].value_counts().head(10))

print("\n🌍 TOP 10 PRODUCTION COUNTRIES:")
print(df['country'].value_counts().head(10))

---

## ⚠️ Step 3: Analyze Missing Values

Missing values are a critical issue in real-world data. Let's understand their patterns.

In [ ]:
# Calculate missing value statistics
missing_stats = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2),
    'Data_Type': df.dtypes.values
}).sort_values('Missing_Count', ascending=False)

print("\n" + "="*100)
print("MISSING VALUES ANALYSIS")
print("="*100)
print(missing_stats.to_string(index=False))

# Visualize missing values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
missing_cols = missing_stats[missing_stats['Missing_Count'] > 0]
ax1.barh(missing_cols['Column'], missing_cols['Missing_Percentage'], color='coral')
ax1.set_xlabel('Percentage Missing (%)')
ax1.set_title('Missing Data by Column', fontsize=12, fontweight='bold')
ax1.invert_yaxis()

# Pie chart of complete vs incomplete records
complete_records = df.dropna().shape[0]
incomplete_records = len(df) - complete_records
ax2.pie([complete_records, incomplete_records], 
        labels=[f'Complete\n({complete_records})', f'Incomplete\n({incomplete_records})'],
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
        startangle=90)
ax2.set_title('Complete vs Incomplete Records', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📌 Summary:")
print(f"  - {complete_records} records are 100% complete")
print(f"  - {incomplete_records} records have at least one missing value")

### 🧐 **Deep Dive: Why Are Directors Missing?**

Let's investigate the missing pattern by content type:

In [ ]:
# Analyze missing directors by content type
print("\nMISSING DIRECTORS BY CONTENT TYPE:")
print("="*50)

missing_by_type = df.groupby('type').agg({
    'director': lambda x: x.isnull().sum(),
    'show_id': 'count'
}).rename(columns={'director': 'Missing', 'show_id': 'Total'})

missing_by_type['Missing_Percentage'] = (missing_by_type['Missing'] / missing_by_type['Total'] * 100).round(1)
print(missing_by_type)

print("\n💡 Insight: TV Shows have MORE missing directors because:")
print("   - Multiple directors per season (incomplete data)")
print("   - Reality/docuseries often don't list directors")
print("   - Data collection inconsistencies")

---

## 🔧 Step 4: Handle Missing Values

### Strategy: Why "Unknown" vs. Delete?

**Option 1: Delete rows with missing values**
- ❌ Loses 2,634 director records (30% of data!)
- ❌ Biased analysis (removes entire genres)

**Option 2: Fill with "Unknown"**
- ✅ Preserves all records
- ✅ Keeps analysis unbiased
- ✅ Explicitly marks missing data
- ✅ Can still analyze the "Unknown" category

**We choose Option 2! 🎯**

In [ ]:
# Create a backup for comparison
df_original = df.copy()

# Fill missing values with "Unknown" placeholders
print("Filling missing values...\n")

fill_map = {
    'director': 'Unknown Director',
    'cast': 'Unknown Cast',
    'country': 'Unknown Country',
    'date_added': 'Unknown Date',
    'rating': 'Unknown Rating',
    'duration': 'Unknown Duration'
}

for column, fill_value in fill_map.items():
    missing_count = df[column].isnull().sum()
    if missing_count > 0:
        df[column] = df[column].fillna(fill_value)
        print(f"✅ {column:15} → Filled {missing_count:4} values with '{fill_value}'")
    else:
        print(f"✅ {column:15} → No missing values")

print("\n" + "="*50)
print(f"Total records with missing values before: {df_original.isnull().any(axis=1).sum()}")
print(f"Total records with missing values after: {df.isnull().any(axis=1).sum()}")
print("="*50)

---

## 🎯 Step 5: Process Multi-Value Fields

Some columns contain multiple comma-separated values. Let's extract and analyze them.

In [ ]:
def split_multi_value_column(series, column_name):
    """
    Split comma-separated values in a pandas Series.
    
    Parameters:
    -----------
    series : pd.Series
        The series to split
    column_name : str
        Name of the column (for logging)
        
    Returns:
    --------
    pd.Series : Exploded and cleaned values
    """
    # Replace 'Unknown' placeholders with empty string
    cleaned = series.str.replace(r'Unknown\s+\w+', '', regex=True)
    
    # Split by comma and explode
    split_values = cleaned.str.split(',').explode().str.strip()
    
    # Remove empty strings
    split_values = split_values[split_values != '']
    
    return split_values

# Process genres
genres = split_multi_value_column(df['listed_in'], 'listed_in')
top_genres = genres.value_counts().head(15)

print("\n" + "="*80)
print("🎭 TOP 15 GENRES ON NETFLIX")
print("="*80)
for rank, (genre, count) in enumerate(top_genres.items(), 1):
    percentage = count / len(genres) * 100
    bar = '█' * int(percentage / 2)
    print(f"{rank:2}. {genre:30} | {count:5} occurrences | {percentage:5.1f}% {bar}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 7))
top_genres.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(top_genres)))
ax.set_xlabel('Number of Titles', fontsize=11, fontweight='bold')
ax.set_ylabel('Genre', fontsize=11, fontweight='bold')
ax.set_title('🎬 Top 15 Genres in Netflix Catalog', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: International & Drama content dominates Netflix's strategy")

In [ ]:
# Process cast information
cast_series = split_multi_value_column(df['cast'], 'cast')
top_actors = cast_series.value_counts().head(10)

print("\n" + "="*80)
print("🎬 TOP 10 MOST FEATURED ACTORS/ACTRESSES")
print("="*80)
for rank, (actor, count) in enumerate(top_actors.items(), 1):
    print(f"{rank:2}. {actor:30} | {count:3} appearances")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
top_actors.plot(kind='bar', ax=ax, color=sns.color_palette('coolwarm', len(top_actors)))
ax.set_xlabel('Actor/Actress', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Appearances', fontsize=11, fontweight='bold')
ax.set_title('🌟 Most Frequently Featured Actors on Netflix', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## 📅 Step 6: Data Type Conversion

Ensure columns are in the correct data types for analysis.

In [ ]:
print("\nDATA TYPE CONVERSIONS:")
print("="*70)

# Convert date_added to datetime (with coercion for 'Unknown Date')
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
nat_count = df['date_added'].isnull().sum()
print(f"✅ date_added      → datetime64[ns] ({nat_count} 'Unknown Date' values became NaT)")

# Ensure release_year is int
df['release_year'] = df['release_year'].astype('int32')
print(f"✅ release_year    → int32")

# Ensure type is categorical (more efficient)
df['type'] = df['type'].astype('category')
print(f"✅ type            → category (Movie, TV Show)")

# Rating as categorical
df['rating'] = df['rating'].astype('category')
print(f"✅ rating          → category")

print("\n" + df.dtypes.to_string())

In [ ]:
# Extract numeric duration for movies
def extract_duration_numeric(row):
    """
    Extract numeric duration from duration column.
    For movies: return minutes
    For TV shows: return number of seasons
    For Unknown: return NaN
    """
    if pd.isna(row['duration']) or row['duration'] == 'Unknown Duration':
        return np.nan
    
    duration_str = str(row['duration']).strip()
    
    try:
        if 'min' in duration_str:
            return float(duration_str.split()[0])
        elif 'Season' in duration_str:
            return float(duration_str.split()[0])
        else:
            return np.nan
    except:
        return np.nan

df['duration_numeric'] = df.apply(extract_duration_numeric, axis=1)

print("\nDURATION STATISTICS:")
print("="*50)
print(f"\nMovies:")
movie_duration = df[df['type'] == 'Movie']['duration_numeric'].describe()
print(f"  Mean duration: {movie_duration['mean']:.1f} minutes")
print(f"  Median duration: {movie_duration['50%']:.1f} minutes")
print(f"  Range: {movie_duration['min']:.0f} - {movie_duration['max']:.0f} minutes")

print(f"\nTV Shows:")
tv_duration = df[df['type'] == 'TV Show']['duration_numeric'].describe()
print(f"  Mean seasons: {tv_duration['mean']:.1f}")
print(f"  Median seasons: {tv_duration['50%']:.1f}")
print(f"  Range: {tv_duration['min']:.0f} - {tv_duration['max']:.0f} seasons")
print(f"  Max found: {df[df['type']=='TV Show']['duration_numeric'].max():.0f} seasons")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Movie duration distribution
movies_data = df[df['type'] == 'Movie']['duration_numeric'].dropna()
axes[0].hist(movies_data, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(movies_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {movies_data.mean():.1f} min')
axes[0].set_xlabel('Duration (minutes)', fontweight='bold')
axes[0].set_ylabel('Number of Movies', fontweight='bold')
axes[0].set_title('📽️ Movie Duration Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# TV show seasons distribution
tv_data = df[df['type'] == 'TV Show']['duration_numeric'].dropna()
axes[1].hist(tv_data, bins=15, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].axvline(tv_data.mean(), color='darkred', linestyle='--', linewidth=2, label=f'Mean: {tv_data.mean():.2f} seasons')
axes[1].set_xlabel('Number of Seasons', fontweight='bold')
axes[1].set_ylabel('Number of TV Shows', fontweight='bold')
axes[1].set_title('📺 TV Show Seasons Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🔄 Step 7: Detect & Remove Duplicates

In [ ]:
print("\nDUPLICATE DETECTION:")
print("="*70)

# Check for complete row duplicates
full_duplicates = df.duplicated().sum()
print(f"\n🔍 Complete row duplicates: {full_duplicates}")

# Check for duplicates based on key columns
key_columns = ['title', 'type', 'release_year']
key_duplicates = df.duplicated(subset=key_columns, keep=False).sum()
print(f"🔍 Potential duplicates (same title + type + year): {key_duplicates}")

if key_duplicates > 0:
    print("\nExample of potential duplicates:")
    dup_example = df[df.duplicated(subset=key_columns, keep=False)].head()
    print(dup_example[['title', 'type', 'release_year', 'country']])
    print("\n💡 Note: These might be valid (different countries, directors, etc.)")

# Remove ONLY complete duplicates
df_before = len(df)
df = df.drop_duplicates(keep='first')
df_after = len(df)
removed = df_before - df_after

print(f"\n✅ Removed {removed} complete duplicate rows")
print(f"   Dataset: {df_before} → {df_after} records")

---

## 📊 Step 8: Exploratory Insights

Now let's explore some interesting patterns in the cleaned data:

In [ ]:
# Content by decade
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'
decade_counts = df.groupby('decade').size().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
decade_counts.plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(decade_counts)), edgecolor='black')
ax.set_xlabel('Decade', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Titles', fontsize=11, fontweight='bold')
ax.set_title('📺 Netflix Content by Decade of Release', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: Majority of Netflix content is from the 2000s-2020s")
print(f"   Latest decade (2020s) has {decade_counts.iloc[-1]} titles (and counting!)")

In [ ]:
# Content growth over time
df_dated = df.dropna(subset=['date_added'])
df_dated['year_added'] = df_dated['date_added'].dt.year
growth = df_dated.groupby('year_added').size()

fig, ax = plt.subplots(figsize=(12, 6))
growth.plot(kind='line', ax=ax, marker='o', linewidth=2.5, markersize=6, color='#e74c3c')
ax.fill_between(growth.index, growth.values, alpha=0.3, color='#e74c3c')
ax.set_xlabel('Year', fontsize=11, fontweight='bold')
ax.set_ylabel('Titles Added', fontsize=11, fontweight='bold')
ax.set_title('📈 Netflix Content Growth by Year Added', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: Netflix accelerated content acquisition from 2018-2021")
print(f"   Peak year: {growth.idxmax()} with {growth.max()} titles added")

In [ ]:
# Rating distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Movies
movie_ratings = df[df['type'] == 'Movie']['rating'].value_counts().head(8)
movie_ratings.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('📽️ Movie Ratings Distribution', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Rating', fontweight='bold')
axes[0].set_ylabel('Count', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# TV Shows
tv_ratings = df[df['type'] == 'TV Show']['rating'].value_counts().head(8)
tv_ratings.plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('📺 TV Show Ratings Distribution', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Rating', fontweight='bold')
axes[1].set_ylabel('Count', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n💡 Insights:")
print(f"   - Movies: More diverse ratings (family-friendly to mature)")
print(f"   - TV Shows: Concentrated on mature audiences (TV-MA)")

---

## ✨ Step 9: Data Quality Summary

Let's create a comprehensive data quality report:

In [ ]:
print("\n" + "="*100)
print("📊 DATA PREPARATION FINAL REPORT")
print("="*100)

print(f"\n📈 DATASET DIMENSIONS:")
print(f"   Original: {df_original.shape[0]} rows × {df_original.shape[1]} columns")
print(f"   Cleaned:  {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Rows removed (duplicates): {df_original.shape[0] - df.shape[0]}")

print(f"\n✅ MISSING VALUE HANDLING:")
print(f"   Records with missing data (original): {df_original.isnull().any(axis=1).sum()}")
print(f"   Records with missing data (cleaned): {df.isnull().any(axis=1).sum()}")
print(f"   Missing values imputed: {df_original.isnull().sum().sum()}")

print(f"\n🎭 CONTENT BREAKDOWN:")
print(f"   Movies: {(df['type']=='Movie').sum():,} ({(df['type']=='Movie').sum()/len(df)*100:.1f}%)")
print(f"   TV Shows: {(df['type']=='TV Show').sum():,} ({(df['type']=='TV Show').sum()/len(df)*100:.1f}%)")

print(f"\n🌍 GEOGRAPHIC COVERAGE:")
print(f"   Unique countries: {df['country'].apply(lambda x: len(str(x).split(';'))).max()}" )
print(f"   Most common: {df['country'].value_counts().index[0]}")

print(f"\n🎪 GENRE COVERAGE:")
genre_count = len(genres.unique())
print(f"   Unique genres: {genre_count}")
print(f"   Most common: {genres.value_counts().index[0]}")

print(f"\n📅 TEMPORAL SPAN:")
print(f"   Release years: {df['release_year'].min():.0f} - {df['release_year'].max():.0f}")
print(f"   Date added range: {df['date_added'].min().date()} to {df['date_added'].max().date()}")

print(f"\n✨ DATA QUALITY SCORE:")
completeness = (1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
print(f"   Completeness: {completeness:.1f}%")
print(f"   Consistency: ✅ (All columns verified)")
print(f"   Duplicates: ✅ (All removed)")
print(f"   Data types: ✅ (Optimized)")

print(f"\n" + "="*100)
print("✨ DATASET READY FOR EXPLORATORY DATA ANALYSIS (EDA)!")
print("="*100)

---

## 🎓 Key Learnings & Best Practices

### ✅ What We Accomplished:
1. **Loaded real-world data** with imperfections
2. **Understood missing data patterns** (directors in TV shows)
3. **Made strategic decisions** (kept vs. delete data)
4. **Processed multi-value fields** (genres, cast)
5. **Converted data types** appropriately
6. **Removed duplicates** safely
7. **Discovered insights** (Netflix's content strategy)

### 💡 Best Practices for Data Cleaning:

| Practice | Why It Matters |
|----------|----------------|
| **Understand missing patterns** | Different reasons for missing data require different treatments |
| **Document all decisions** | Others need to understand your choices |
| **Keep original data** | Always maintain a backup for comparison |
| **Validate assumptions** | Check if your imputation strategy makes sense |
| **Iterate & improve** | Data cleaning is rarely one-shot; you learn as you go |
| **Visualize before & after** | See the impact of your cleaning steps |
| **Test edge cases** | Don't assume all data is "normal" |

### 🚀 Next Steps (EDA Phase):
- Statistical analysis of content characteristics
- Correlation analysis between features
- Natural language processing on descriptions
- Time series analysis of content trends
- Predictive modeling for content success

---

## 🤔 Challenge Questions for You:

1. **Why might Netflix have more movies than TV shows in the catalog?**
2. **What could we learn from the "Unknown Director" entries?**
3. **How would you analyze the relationship between release year and rating?**
4. **Can you predict whether a title is a movie or TV show based on other features?**
5. **What's the most interesting pattern you found in the data?**

---

In [ ]:
# Optional: Save the cleaned dataset for next phase
# df.to_csv('netflix_cleaned.csv', index=False)
# print("✅ Cleaned dataset saved to 'netflix_cleaned.csv'")